# Eminem Lyric Generator - RNN with PyTorch

This notebook implements a character-level RNN (using a GRU) to generate lyrics in the style of Eminem. 

### Steps:
1. **Preprocessing**: Load text and map characters to integers.
2. **Data Loading**: Create sequences for training.
3. **Model Architecture**: Define a multi-layer RNN (GRU).
4. **Training**: Implement the training loop.
5. **Generation**: Function to generate new text from a seed.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import os

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Preprocessing
We load the cleaned lyrics and create character mappings.

In [ ]:
path_to_file = '../data/cleaned_eminem.txt'
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')

chars = sorted(list(set(text)))
char_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_char = {i: ch for i, ch in enumerate(chars)}
vocab_size = len(chars)

print(f"Total characters: {len(text)}")
print(f"Unique characters: {vocab_size}")

## 2. Model Architecture
We use an Embedding layer, a GRU (better than basic RNN for sequences), and a Linear output layer.

In [ ]:
class LyricRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super(LyricRNN, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, x, h):
        x = self.embed(x)
        out, h = self.gru(x, h)
        out = self.fc(out.reshape(out.size(0) * out.size(1), out.size(2)))
        return out, h

# Hyperparameters
embed_size = 256
hidden_size = 512
num_layers = 2
seq_length = 100
batch_size = 64
learning_rate = 0.001
num_epochs = 20

model = LyricRNN(vocab_size, embed_size, hidden_size, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

## 3. Training Loop
We slice the text into sequences and train the model to predict the next character.

In [ ]:
# Prepare encoded data
encoded_text = np.array([char_to_int[ch] for ch in text])

def get_batches(data, batch_size, seq_length):
    n_batches = len(data) // (batch_size * seq_length)
    data = data[:n_batches * batch_size * seq_length]
    data = data.reshape((batch_size, -1))
    
    for n in range(0, data.shape[1], seq_length):
        x = data[:, n:n+seq_length]
        y = np.zeros_like(x)
        try:
            y[:, :-1], y[:, -1] = x[:, 1:], data[:, n+seq_length]
        except IndexError:
            y[:, :-1], y[:, -1] = x[:, 1:], data[:, 0]
        yield torch.tensor(x), torch.tensor(y)

# Training
model.train()
for epoch in range(num_epochs):
    h = None # Initial hidden state
    for i, (inputs, targets) in enumerate(get_batches(encoded_text, batch_size, seq_length)):
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Forward pass
        outputs, h = model(inputs, h)
        # Detach hidden state to prevent backpropagating through the entire history
        h = h.detach()
        
        loss = criterion(outputs, targets.reshape(-1))
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (i+1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}], Loss: {loss.item():.4f}')

## 4. Generation
Function to sample from the model.

In [ ]:
def generate(model, start_str='Look', length=200, temperature=0.7):
    model.eval()
    chars = [ch for ch in start_str]
    input_seq = torch.tensor([[char_to_int[ch] for ch in start_str]]).to(device)
    h = None
    
    for _ in range(length):
        output, h = model(input_seq, h)
        
        # Sample from the network as a multinomial distribution
        output_dist = output[-1].data.cpu().exp()
        top_ch = torch.multinomial(output_dist, 1)[0].item()
        
        chars.append(int_to_char[top_ch])
        input_seq = torch.tensor([[top_ch]]).to(device)
        
    return ''.join(chars)

print(generate(model, start_str="I'm beginning to feel like"))